# Analysis outline

See [README.md](README.md).

# Set up

Run the following cell.  It loads some required functions.

In [36]:
run kondrashov

# Search for human sequences

Below we try to retrieve human sequences for each gene programatically.  We
choose the [RefSeq Select](https://www.ncbi.nlm.nih.gov/refseq/refseq_select/) database.

In [ ]:
# genes in Kondrashov et al. (2002) are already listed in kondrashov.py
print(loci)

['ABCD1', 'ALPL', 'AR', 'ATP7B', 'BTK', 'CASR', 'CBS', 'CFTR', 'CYBB', 'F7', 'F8', 'F9', 'G6PD', 'GALT', 'GBA', 'GJB1', 'HBB', 'HPRT1', 'IL2RG', 'KCNH2', 'KCNQ1', 'L1CAM', 'LDLR', 'MPZ', 'MYH7', 'TYR', 'PAH', 'PMM2', 'RHO', 'TP53', 'TTR', 'VWF']


In [ ]:
genes = {}
for locus in loci:
    query = f"Homo sapiens[Organism] AND {locus}[Gene Name] AND refseq_select[Filter]"
    handle = Entrez.esearch(db="nucleotide", term=query, retmax=100)
    record = Entrez.read(handle)
    handle.close()
    print(f"{locus}: {record['Count']} ->", record['IdList'][:4])
    genes.update({locus: record['IdList']})

ABCD1: 1 -> ['7262393']
ALPL: 1 -> ['116734717']
AR: 3 -> ['21322252', '4502199', '4502049']
ATP7B: 1 -> ['55743071']
BTK: 1 -> ['4557377']
CASR: 1 -> ['1677537480']
CBS: 1 -> ['4557415']
CFTR: 1 -> ['90421313']
CYBB: 1 -> ['6996021']
F7: 1 -> ['10518503']
F8: 1 -> ['4503647']
F9: 1 -> ['4503649']
G6PD: 1 -> ['1334575534']
GALT: 1 -> ['22165416']
GBA: 1 -> ['54607043']
GJB1: 1 -> ['4504005']
HBB: 1 -> ['4504349']
HPRT1: 1 -> ['4504483']
IL2RG: 1 -> ['4557882']
KCNH2: 1 -> ['4557729']
KCNQ1: 1 -> ['32479527']
L1CAM: 1 -> ['497239609']
LDLR: 1 -> ['4504975']
MPZ: 1 -> ['295391071']
MYH7: 1 -> ['115496169']
TYR: 1 -> ['4507753']
PAH: 1 -> ['4557819']
PMM2: 1 -> ['4557839']
RHO: 2 -> ['4506527', '7656900']
TP53: 1 -> ['120407068']
TTR: 1 -> ['4507725']
VWF: 1 -> ['1813372009']


Two of the genes, AR and RHO, have more than one RefSeq Select transcripts.
Let's look at them more closely.

* AR: only the first sequence is valid.

* RHO: only the second sequence is valid.

In [98]:
# AR
for i in genes['AR']:
    stream = Entrez.efetch(db="nucleotide", id=i, rettype="gb", retmode="text")
    record2 = SeqIO.read(stream, "genbank")
    stream.close()
    print(f"ID: {i}")
    print(f"{record2.id}: {record2.description}\nLength: {len(record2.seq)} nt\n")

ID: 1654124212
NM_000044.6: Homo sapiens androgen receptor (AR), transcript variant 1, mRNA
Length: 10667 nt

ID: 1519245710
NM_001657.4: Homo sapiens amphiregulin (AREG), mRNA
Length: 1234 nt

ID: 1519243240
NM_001628.4: Homo sapiens aldo-keto reductase family 1 member B (AKR1B1), transcript variant 1, mRNA
Length: 1361 nt



In [99]:
# remove wrong sequences
del genes['AR'][2]
del genes['AR'][1]
genes['AR']

['1654124212']

In [100]:
# RHO
for i in genes['RHO']:
    stream = Entrez.efetch(db="nucleotide", id=i, rettype="gb", retmode="text")
    record2 = SeqIO.read(stream, "genbank")
    stream.close()
    print(f"ID: {i}")
    print(f"{record2.id}: {record2.description}\nLength: {len(record2.seq)} nt\n")

ID: 1519316345
NM_014578.4: Homo sapiens ras homolog family member D (RHOD), transcript variant 1, mRNA
Length: 1104 nt

ID: 169808383
NM_000539.3: Homo sapiens rhodopsin (RHO), mRNA
Length: 2768 nt



In [101]:
# remove wrong sequence
del genes['RHO'][0]
genes['RHO']

['169808383']

In [102]:
for gene in genes:
    stream = Entrez.efetch(db="nucleotide", id=genes[gene][0], rettype="gb", retmode="text")
    record2 = SeqIO.read(stream, "genbank")
    stream.close()
    genes[gene].append(record2.id)
    print(gene, genes[gene])

ABCD1 ['1519313321', 'NM_000033.4']
ALPL ['1519315936', 'NM_000478.6']
AR ['1654124212', 'NM_000044.6']
ATP7B ['1677498587', 'NM_000053.4']
BTK ['1780222528', 'NM_000061.3']
CASR ['1677537479', 'NM_000388.4']
CBS ['1780222567', 'NM_000071.3']
CFTR ['1732746288', 'NM_000492.4']
CYBB ['1732746192', 'NM_000397.4']
F7 ['1779521762', 'NM_019616.4']
F8 ['1812229661', 'NM_000132.4']
F9 ['1732746198', 'NM_000133.4']
G6PD ['1497290737', 'NM_001360016.2']
GALT ['1677502190', 'NM_000155.4']
GBA ['1519244100', 'NM_000157.4']
GJB1 ['1606717073', 'NM_000166.6']
HBB ['1401724401', 'NM_000518.5']
HPRT1 ['1519243265', 'NM_000194.3']
IL2RG ['1780222514', 'NM_000206.3']
KCNH2 ['1732746325', 'NM_000238.4']
KCNQ1 ['1732746161', 'NM_000218.3']
L1CAM ['1653961419', 'NM_001278116.2']
LDLR ['1732746181', 'NM_000527.5']
MPZ ['1519245315', 'NM_000530.8']
MYH7 ['1519242569', 'NM_000257.4']
TYR ['1519243869', 'NM_000372.5']
PAH ['1519244862', 'NM_000277.3']
PMM2 ['1519243247', 'NM_000303.3']
RHO ['169808383', 'NM_

In [141]:
# pull protein ID
handle_link = Entrez.elink(dbfrom="nuccore", db="protein", id="NM_000552.5")
link_record = Entrez.read(handle_link)
protein_id = link_record[0]["LinkSetDb"][0]["Link"][0]["Id"]
protein_id

'1813372009'

In [ ]:
# pull protein sequence ID
stream = Entrez.efetch(db="protein", id='1813372009', rettype="gb", retmode="text")
record2 = SeqIO.read(stream, "genbank")
stream.close()
record2.id

'NP_000543.3'

# Search for orthologs

Primates are represented by [Registry Number
txid9443](https://www.ncbi.nlm.nih.gov/mesh/68011323).

Searching in the `Gene` database appears to retrieve the sequences that can be
obtained by doing the following:

* In the [NCBI query page](https://www.ncbi.nlm.nih.gov/labs/gquery/) search for
  `Homo sapiens {Gene Name}` (remove braces).  

* When the search results come up click on `Orthologs`.  

* Under `Selected taxa` write **Primates**. 

This was the procedure used in Hannah Esopenko's analysis.

In [91]:
orth = {}
for locus in loci:
    query = f"{locus}[Gene Name] AND txid9443[Organism]"
    handle = Entrez.esearch(db="gene", term=query, retmax=100)
    record = Entrez.read(handle)
    handle.close()
    print(f"{locus}: {record['Count']} ->", record['IdList'][:4])
    orth.update({locus: record['IdList']})

ABCD1: 35 -> ['215', '6367', '696794', '465923']
ALPL: 35 -> ['249', '717809', '456600', '100401643']
AR: 39 -> ['367', '374', '231', '574293']
ATP7B: 34 -> ['540', '713781', '452734', '101866907']
BTK: 34 -> ['695', '703000', '465759', '102123534']
CASR: 35 -> ['846', '714441', '460628', '101867346']
CBS: 32 -> ['875', '736626', '100423684', '100414619']
CFTR: 37 -> ['1080', '574346', '463674', '100137035']
CYBB: 34 -> ['1536', '696542', '465566', '102122418']
F7: 32 -> ['2155', '721933', '744785', '100410381']
F8: 33 -> ['2157', '100424151', '473838', '100393161']
F9: 34 -> ['2158', '695578', '465887', '102146175']
G6PD: 33 -> ['2539', '701137', '743041', '100388810']
GALT: 38 -> ['2592', '145173', '473219', '106993709']
GBA: 28 -> ['2629', '449571', '719103', '123634063']
GJB1: 35 -> ['2705', '706327', '465696', '102128042']
HBB: 25 -> ['3043', '450978', '101926697', '715559']
HPRT1: 34 -> ['3251', '735894', '101867079', '709186']
IL2RG: 36 -> ['3561', '641338', '101059843', '102144

In [ ]:
# MPZ: 35 -> ['4359', '719969', '747431', '102115938']
handle = Entrez.efetch(db="gene", id='4359', rettype="xml", retmode="text")
record = x2d.parse(handle.read().decode('utf-8'))
handle.close()
record

{'Entrezgene-Set': {'Entrezgene': {'Entrezgene_track-info': {'Gene-track': {'Gene-track_geneid': '4359',
     'Gene-track_status': {'@value': 'live', '#text': '0'},
     'Gene-track_create-date': {'Date': {'Date_std': {'Date-std': {'Date-std_year': '1998',
         'Date-std_month': '8',
         'Date-std_day': '20'}}}},
     'Gene-track_update-date': {'Date': {'Date_std': {'Date-std': {'Date-std_year': '2026',
         'Date-std_month': '5',
         'Date-std_day': '19',
         'Date-std_hour': '8',
         'Date-std_minute': '32',
         'Date-std_second': '0'}}}}}},
   'Entrezgene_type': {'@value': 'protein-coding', '#text': '6'},
   'Entrezgene_source': {'BioSource': {'BioSource_genome': {'@value': 'genomic',
      '#text': '1'},
     'BioSource_origin': {'@value': 'natural', '#text': '1'},
     'BioSource_org': {'Org-ref': {'Org-ref_taxname': 'Homo sapiens',
       'Org-ref_common': 'human',
       'Org-ref_db': {'Dbtag': {'Dbtag_db': 'taxon',
         'Dbtag_tag': {'Object

In [118]:
record['Entrezgene-Set']['Entrezgene'].keys()

dict_keys(['Entrezgene_track-info', 'Entrezgene_type', 'Entrezgene_source', 'Entrezgene_gene', 'Entrezgene_prot', 'Entrezgene_summary', 'Entrezgene_location', 'Entrezgene_gene-source', 'Entrezgene_locus', 'Entrezgene_properties', 'Entrezgene_comments', 'Entrezgene_unique-keys', 'Entrezgene_xtra-index-terms', 'Entrezgene_xtra-properties'])

In [133]:
record['Entrezgene-Set']['Entrezgene']['Entrezgene_locus']['Gene-commentary'][0]['Gene-commentary_accession']

'NC_000001'

In [134]:
record['Entrezgene-Set']['Entrezgene']['Entrezgene_locus']['Gene-commentary'][0]['Gene-commentary_version']

'11'

In [135]:
get_transcript('NC_000001.11')

{'GBSeq_locus': 'NC_000001', 'GBSeq_length': '248956422', 'GBSeq_strandedness': 'double', 'GBSeq_moltype': 'DNA', 'GBSeq_topology': 'linear', 'GBSeq_division': 'CON', 'GBSeq_update-date': '06-AUG-2025', 'GBSeq_create-date': '29-AUG-2002', 'GBSeq_definition': 'Homo sapiens chromosome 1, GRCh38.p14 Primary Assembly', 'GBSeq_primary-accession': 'NC_000001', 'GBSeq_accession-version': 'NC_000001.11', 'GBSeq_other-seqids': ['gnl|ASM:GCF_000001305|1', 'ref|NC_000001.11|', 'gpp|GPC_000001293.1|', 'gnl|NCBI_GENOMES|1', 'gi|568815597'], 'GBSeq_project': 'PRJNA168', 'GBSeq_keywords': ['RefSeq'], 'GBSeq_source': 'Homo sapiens (human)', 'GBSeq_organism': 'Homo sapiens', 'GBSeq_taxonomy': 'Eukaryota; Metazoa; Chordata; Craniata; Vertebrata; Euteleostomi; Mammalia; Eutheria; Euarchontoglires; Primates; Haplorrhini; Catarrhini; Hominidae; Homo', 'GBSeq_references': [{'GBReference_reference': '1', 'GBReference_position': '1..248956422', 'GBReference_authors': ['Gregory,S.G.', 'Barlow,K.F.', 'McLay,K.E

In [ ]:
for i in record['Entrezgene-Set']['Entrezgene'].keys():
    print(i)
    print()

dict_keys(['Entrezgene_track-info', 'Entrezgene_type', 'Entrezgene_source', 'Entrezgene_gene', 'Entrezgene_prot', 'Entrezgene_summary', 'Entrezgene_location', 'Entrezgene_gene-source', 'Entrezgene_locus', 'Entrezgene_properties', 'Entrezgene_comments', 'Entrezgene_unique-keys', 'Entrezgene_xtra-index-terms', 'Entrezgene_xtra-properties'])

In [113]:
record

{'Entrezgene-Set': {'Entrezgene': {'Entrezgene_track-info': {'Gene-track': {'Gene-track_geneid': '105822916',
     'Gene-track_status': {'@value': 'live', '#text': '0'},
     'Gene-track_create-date': {'Date': {'Date_std': {'Date-std': {'Date-std_year': '2015',
         'Date-std_month': '6',
         'Date-std_day': '1'}}}},
     'Gene-track_update-date': {'Date': {'Date_std': {'Date-std': {'Date-std_year': '2026',
         'Date-std_month': '4',
         'Date-std_day': '8',
         'Date-std_hour': '16',
         'Date-std_minute': '2',
         'Date-std_second': '36'}}}}}},
   'Entrezgene_type': {'@value': 'protein-coding', '#text': '6'},
   'Entrezgene_source': {'BioSource': {'BioSource_genome': {'@value': 'genomic',
      '#text': '1'},
     'BioSource_origin': {'@value': 'natural', '#text': '1'},
     'BioSource_org': {'Org-ref': {'Org-ref_taxname': 'Propithecus coquereli',
       'Org-ref_common': "Coquerel's sifaka",
       'Org-ref_db': {'Dbtag': {'Dbtag_db': 'taxon',
     

In [114]:
orth['CBS']

['875', '736626', '100423684', '100414619', '138387809', '128568008', '126950623', '123630779', '117095839', '116550763', '116482221', '112620305', '111526481', '108529529', '108311214', '105822916', '105713305', '105575589', '104677999', '101007629', '100974696', '100948655', '100597744', '129483030', '129022620', '105879116', '103221104', '102115084', '101149756', '101033999', '100459501', '105472571']